# Adaptive Filtered-COS: Fang–Oosterlee Test Cases

**Columbia University · MAFN · MATH 5030 · Spring 2026**

This notebook benchmarks three COS variants on the canonical test cases from
Fang & Oosterlee (2008):

| Method | Description |
|--------|-------------|
| **Vanilla COS** | Cumulant-based interval $[a,b]$, fixed $L=10$, no filter |
| **Junike COS** | Tolerance-driven interval widening, no filter |
| **Adaptive filtered COS** | Junike interval + deterministic filter search |

For each test case we show:
- Error vs $N$ convergence for vanilla and Junike COS;
- The adaptive selector result (auto-chosen $N$, filter, error, runtime).

**References**
- Fang & Oosterlee (2008), *A novel pricing method for European options based on Fourier-cosine series expansions*, SIAM J. Sci. Comput.
- Junike & Pankrashkin (2022), *Precise option pricing by the COS method*, Applied Mathematics and Computation.
- Ruijter, Versteegh & Oosterlee (2015), *On the application of spectral filters in a Fourier option pricing technique*, J. Computational Finance.


In [ ]:
%pip install -q -U fourier-option-pricer


In [ ]:
import pathlib, sys, importlib, importlib.util
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if importlib.util.find_spec('foureng') is None:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'fourier-option-pricer'], check=True)
    importlib.invalidate_caches()

from foureng.iv.implied_vol    import BSInputs, bs_price_from_fwd
from foureng.models.base       import ForwardSpec
from foureng.models.bsm        import BsmParams
from foureng.models.heston     import HestonParams, heston_cumulants
from foureng.models.variance_gamma import VGParams
from foureng.models.cgmy       import CgmyParams
from foureng.pipeline          import price_strip
from foureng.pricers.cos       import cos_auto_grid, recommended_cos_policy
from foureng.utils.grids       import COSGridPolicy

# Extension modules not in PyPI v0.2.0 -- try editable install fallback.
def _try_ext():
    try:
        from foureng.utils.spectral_filters import COSFilterSpec           # noqa
        from foureng.experiments.cos_filter_grid_search import (           # noqa
            FilterGridCandidate, run_filtered_cos_grid_search,
            select_fastest_under_tolerance,
        )
        return True
    except (ImportError, ModuleNotFoundError):
        return False

if not _try_ext():
    _root = pathlib.Path('.').resolve()
    for _ in range(6):
        if (_root / 'pyproject.toml').exists():
            break
        _root = _root.parent
    for _k in [k for k in sys.modules if k == 'foureng' or k.startswith('foureng.')]:
        del sys.modules[_k]
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    importlib.invalidate_caches()
    if not _try_ext():
        raise ImportError('Run: pip install -e . from the repo root.')

from foureng.utils.spectral_filters import COSFilterSpec
from foureng.experiments.cos_filter_grid_search import (
    FilterGridCandidate, run_filtered_cos_grid_search, select_fastest_under_tolerance,
)

# ── shared helpers ────────────────────────────────────────────────────────
def sci(x):
    return '--' if pd.isna(x) else f'{x:.2e}'

def junike_policy(model, N=None):
    """Return a Junike-style COSGridPolicy with optional fixed N."""
    trunc = 'heuristic' if model == 'vg' else 'tolerance'
    kw = dict(mode='benchmark', truncation=trunc, centered=True, eps_trunc=1e-10)
    if N is not None:
        kw['fixed_N'] = N
    else:
        kw['dx_target'] = 0.003
    return COSGridPolicy(**kw)

def adaptive_candidates(model):
    pol = junike_policy(model)
    return [
        FilterGridCandidate('no_filter',  pol, None),
        FilterGridCandidate('fejer',      pol, COSFilterSpec('fejer')),
        FilterGridCandidate('lanczos',    pol, COSFilterSpec('lanczos')),
        FilterGridCandidate('raised_cos', pol, COSFilterSpec('raised_cosine')),
        FilterGridCandidate('exp_p4',     pol, COSFilterSpec('exponential', order=4)),
        FilterGridCandidate('exp_p8',     pol, COSFilterSpec('exponential', order=8)),
        FilterGridCandidate('exp_p12',    pol, COSFilterSpec('exponential', order=12)),
    ]

def n_sweep(model, fwd, params, strikes, ref, *, Ns, cumulants):
    """Return DataFrame of vanilla vs Junike errors at each N."""
    rows = []
    for N in Ns:
        g   = cos_auto_grid(cumulants, N=N, L=10.0)
        p_v = price_strip(model, 'cos', strikes, fwd, params, grid=g)
        p_j = price_strip(model, 'cos_improved', strikes, fwd, params,
                          grid=junike_policy(model, N=N))
        rows.append(dict(
            N=N,
            vanilla_err=float(np.max(np.abs(np.asarray(p_v) - ref))),
            junike_err =float(np.max(np.abs(np.asarray(p_j) - ref))),
        ))
    return pd.DataFrame(rows)

def run_adaptive(model, fwd, params, strikes, ref, tol=1e-6):
    """Run grid search and return (best_row, full_df)."""
    df = run_filtered_cos_grid_search(
        model=model, strikes=strikes, fwd=fwd, params=params,
        reference=ref, candidates=adaptive_candidates(model),
        tol=tol, n_repeat=3,
    )
    return select_fastest_under_tolerance(df, tol=tol), df

def plot_convergence(ax, sweep_df, *, title, tol=1e-6, adaptive_err=None,
                     adaptive_label='adaptive'):
    ax.plot(sweep_df['N'], sweep_df['vanilla_err'], 'o-', color='#888',
            label='Vanilla COS')
    ax.plot(sweep_df['N'], sweep_df['junike_err'],  's-', color='steelblue',
            label='Junike COS')
    if adaptive_err is not None:
        ax.axhline(adaptive_err, color='crimson', ls='--', lw=1.5,
                   label=f'{adaptive_label} (auto-N)')
    ax.axhline(tol, color='black', ls='-.', lw=0.9, alpha=0.5,
               label=f'tol={tol:.0e}')
    ax.set_xscale('log', base=2); ax.set_yscale('log')
    ax.set_xlabel('N (number of COS terms)')
    ax.set_ylabel('max|error|')
    ax.set_title(title)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(True, which='both', ls='--', alpha=0.3)

TOL = 1e-6
NS  = [16, 32, 64, 128, 256, 512, 1024]
print('Setup complete.')


## §1 · Black-Scholes: FO2008 Test Case 1

Parameters: $S_0 = 100$, $r = 0.1$, $q = 0$, $T = 1$, $\sigma = 0.25$. Reference: Black-Scholes closed form.

BSM is a smooth-CF model: all three COS variants should converge rapidly, and filtering should be unnecessary (adaptive selects no-filter).


In [ ]:
BSM_FWD    = ForwardSpec(S0=100, r=0.1, q=0.0, T=1.0)
BSM_PARAMS = BsmParams(sigma=0.25)
BSM_K      = np.array([80., 85., 90., 95., 100., 105., 110., 115., 120.])
BSM_CUMS   = type('C', (), {'c1': BSM_FWD.F0, 'c2': BSM_PARAMS.sigma**2 * BSM_FWD.T,
                              'c4': 0.0})()

BSM_REF = np.array([
    bs_price_from_fwd(
        BSM_PARAMS.sigma,
        BSInputs(F0=BSM_FWD.F0, K=float(k), T=BSM_FWD.T,
                 r=BSM_FWD.r, q=BSM_FWD.q, is_call=True),
    ) for k in BSM_K
])

from foureng.models.bsm import bsm_cumulants
BSM_CUMS = bsm_cumulants(BSM_FWD, BSM_PARAMS)

bsm_sweep = n_sweep('bsm', BSM_FWD, BSM_PARAMS, BSM_K, BSM_REF,
                     Ns=NS, cumulants=BSM_CUMS)
bsm_best, bsm_df = run_adaptive('bsm', BSM_FWD, BSM_PARAMS, BSM_K, BSM_REF)

print('BSM convergence:')
print(bsm_sweep.to_string(index=False))
print(f"\nAdaptive: filter={bsm_best['filter']}, "
      f"error={bsm_best['max_abs_err']:.2e}, "
      f"runtime={bsm_best['runtime_ms']:.2f} ms")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_convergence(axes[0], bsm_sweep, title='BSM: error vs N',
                 tol=TOL, adaptive_err=float(bsm_best['max_abs_err']),
                 adaptive_label=f"adaptive ({bsm_best['filter']})")

# Scatter: all adaptive candidates
_ok = bsm_df[bsm_df['status'] == 'ok']
_cmap = {'none':'steelblue','fejer':'darkorange','lanczos':'green',
          'raised_cosine':'purple','exponential':'crimson'}
_seen = set()
for _, row in _ok.iterrows():
    lbl = row['filter'] if row['filter'] not in _seen else ''
    _seen.add(row['filter'])
    axes[1].scatter(row['runtime_ms'], max(row['max_abs_err'], 1e-16),
                    c=_cmap.get(row['filter'], 'grey'), s=55, alpha=0.7, label=lbl)
axes[1].scatter(float(bsm_best['runtime_ms']), max(float(bsm_best['max_abs_err']), 1e-16),
                c='gold', marker='*', s=300, zorder=5, label='Selected')
axes[1].axhline(TOL, color='black', ls='-.', lw=0.9, alpha=0.5)
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('runtime (ms)'); axes[1].set_ylabel('max|error|')
axes[1].set_title('BSM: adaptive candidate space')
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(dict(zip(labels,handles)).values(),
               dict(zip(labels,handles)).keys(), fontsize=7)
axes[1].grid(True, which='both', ls='--', alpha=0.3)

fig.tight_layout()
plt.show()


## §2 · Heston: FO2008 Table 2 + Table 5 Stress Case

**Strip (Table 2 parameters):** $S_0=100$, $r=0.1$, $q=0$, $T=1$; $\kappa=1.5$, $\bar v=0.04$, $\nu=0.3$, $\rho=-0.9$, $v_0=0.04$. Reference: PyFENG FFT.

**Stress case (Table 5):** $T=10$ long-maturity ATM. Reference: published value $22.318945791$. This is where Junike truncation gives the largest gain over vanilla COS (demonstrated in §4); filtering adds a smaller secondary benefit.


In [ ]:
# Table 2 strip
H_FWD    = ForwardSpec(S0=100, r=0.1, q=0.0, T=1.0)
H_PARAMS = HestonParams(kappa=1.5, theta=0.04, nu=0.3, rho=-0.9, v0=0.04)
H_K      = np.array([80., 90., 100., 110., 120.])
H_CUMS   = heston_cumulants(H_FWD, H_PARAMS)
H_REF    = np.asarray(price_strip('heston', 'pyfeng_fft', H_K, H_FWD, H_PARAMS),
                       dtype=float)

h_sweep = n_sweep('heston', H_FWD, H_PARAMS, H_K, H_REF,
                   Ns=NS, cumulants=H_CUMS)
h_best, h_df = run_adaptive('heston', H_FWD, H_PARAMS, H_K, H_REF)

print('Heston T=1 convergence:')
print(h_sweep.to_string(index=False))
print(f"\nAdaptive: filter={h_best['filter']}, "
      f"error={h_best['max_abs_err']:.2e}, "
      f"runtime={h_best['runtime_ms']:.2f} ms")

# Table 5 stress case
H10_FWD    = ForwardSpec(S0=100, r=0.0, q=0.0, T=10.0)
H10_PARAMS = HestonParams(kappa=1.5768, theta=0.0398, nu=0.5751, rho=-0.5711, v0=0.0175)
H10_K      = np.array([100.0])
H10_REF    = np.array([22.318945791])
H10_CUMS   = heston_cumulants(H10_FWD, H10_PARAMS)

h10_sweep = n_sweep('heston', H10_FWD, H10_PARAMS, H10_K, H10_REF,
                    Ns=NS, cumulants=H10_CUMS)
h10_best, h10_df = run_adaptive('heston', H10_FWD, H10_PARAMS, H10_K, H10_REF)

print('\nHeston T=10 (FO Table 5) convergence:')
print(h10_sweep.to_string(index=False))
print(f"\nAdaptive: filter={h10_best['filter']}, "
      f"error={h10_best['max_abs_err']:.2e}, "
      f"runtime={h10_best['runtime_ms']:.2f} ms")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_convergence(axes[0], h_sweep, title='Heston T=1: error vs N',
                 tol=TOL, adaptive_err=float(h_best['max_abs_err']),
                 adaptive_label=f"adaptive ({h_best['filter']})")

plot_convergence(axes[1], h10_sweep, title='Heston T=10 (FO Table 5): error vs N',
                 tol=TOL, adaptive_err=float(h10_best['max_abs_err']),
                 adaptive_label=f"adaptive ({h10_best['filter']})")

fig.tight_layout()
plt.show()


## §3 · Jump Models: VG and CGMY

Pure-jump Lévy models have characteristic functions that decay more slowly in frequency and log-return densities that are more peaked at short maturities. Both effects amplify finite-series oscillatory error, making them the prime candidates for spectral filtering.

**VG (short maturity):** $T=0.1$, Madan–Carr–Chang parameters ($\sigma=0.12$, $\nu=0.2$, $\theta=-0.14$). The density is highly peaked; vanilla COS converges slowly.

**CGMY:** $T=0.25$, $C=1$, $G=5$, $M=10$, $Y=0.5$ (infinite activity, finite variation). CF oscillates; Junike truncation helps but oscillatory error can persist at moderate $N$.

*Note on VG truncation:* `recommended_cos_policy` uses `truncation='heuristic'` for VG (not `'tolerance'`), because VG's tail is light enough that cumulant sizing is reliable. The Junike N-sweep below respects this model-specific choice.


In [ ]:
# VG short maturity
VG_FWD    = ForwardSpec(S0=100, r=0.1, q=0.0, T=0.1)
VG_PARAMS = VGParams(sigma=0.12, nu=0.2, theta=-0.14)
VG_K      = np.linspace(70, 130, 13)
VG_REF    = np.asarray(price_strip('vg', 'pyfeng_fft', VG_K, VG_FWD, VG_PARAMS),
                        dtype=float)

from foureng.models.variance_gamma import vg_cumulants
VG_CUMS = vg_cumulants(VG_FWD, VG_PARAMS)

vg_sweep = n_sweep('vg', VG_FWD, VG_PARAMS, VG_K, VG_REF,
                    Ns=NS, cumulants=VG_CUMS)
vg_best, vg_df = run_adaptive('vg', VG_FWD, VG_PARAMS, VG_K, VG_REF)

print('VG T=0.1 convergence:')
print(vg_sweep.to_string(index=False))
print(f"\nAdaptive: filter={vg_best['filter']}, "
      f"error={vg_best['max_abs_err']:.2e}, "
      f"runtime={vg_best['runtime_ms']:.2f} ms")

# CGMY
CG_FWD    = ForwardSpec(S0=100, r=0.04, q=0.0, T=0.25)
CG_PARAMS = CgmyParams(C=1.0, G=5.0, M=10.0, Y=0.5)
CG_K      = np.linspace(80, 120, 9)
CG_REF    = np.asarray(price_strip('cgmy', 'pyfeng_fft', CG_K, CG_FWD, CG_PARAMS),
                        dtype=float)

from foureng.models.cgmy import cgmy_cumulants
CG_CUMS = cgmy_cumulants(CG_FWD, CG_PARAMS)

cg_sweep = n_sweep('cgmy', CG_FWD, CG_PARAMS, CG_K, CG_REF,
                    Ns=NS, cumulants=CG_CUMS)
cg_best, cg_df = run_adaptive('cgmy', CG_FWD, CG_PARAMS, CG_K, CG_REF)

print('\nCGMY T=0.25 convergence:')
print(cg_sweep.to_string(index=False))
print(f"\nAdaptive: filter={cg_best['filter']}, "
      f"error={cg_best['max_abs_err']:.2e}, "
      f"runtime={cg_best['runtime_ms']:.2f} ms")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_convergence(axes[0], vg_sweep, title='VG T=0.1: error vs N',
                 tol=TOL, adaptive_err=float(vg_best['max_abs_err']),
                 adaptive_label=f"adaptive ({vg_best['filter']})")

plot_convergence(axes[1], cg_sweep, title='CGMY T=0.25: error vs N',
                 tol=TOL, adaptive_err=float(cg_best['max_abs_err']),
                 adaptive_label=f"adaptive ({cg_best['filter']})")

fig.tight_layout()
plt.show()


## §4 · Summary

Collected results across all FO2008 test cases.


In [ ]:
cases_summary = [
    ('BSM T=1',       bsm_sweep,  bsm_best),
    ('Heston T=1',    h_sweep,    h_best),
    ('Heston T=10',   h10_sweep,  h10_best),
    ('VG T=0.1',      vg_sweep,   vg_best),
    ('CGMY T=0.25',   cg_sweep,   cg_best),
]

rows = []
for name, sw, best in cases_summary:
    n256 = sw[sw['N'] == 256].iloc[0] if 256 in sw['N'].values else sw.iloc[-1]
    rows.append({
        'case':               name,
        'vanilla N=256 err':  sci(n256['vanilla_err']),
        'junike N=256 err':   sci(n256['junike_err']),
        'adaptive err':       sci(float(best['max_abs_err'])),
        'adaptive filter':    best['filter'],
        'adaptive rt (ms)':   f"{best['runtime_ms']:.2f}",
    })

summary_df = pd.DataFrame(rows)
display(summary_df.to_string(index=False))
